# 04 — Activity Curves (Full Scale)

Verifies and inspects the corrected `a_curves_corrected.csv` built in
`data/full_scale_run/03_pollinator_occurrences.ipynb`.

This notebook does not rebuild the curves — it loads the pre-built file,
checks coverage and shape, and produces summary visualizations.

**Input:** `a_curves_corrected.csv` (25,466 species × 52 weeks)

**Critical reminder:** GBIF-derived activity curves reflect observation
timing, not true pollinator phenology. The observation-density bias
documented in the scope and limitations applies fully here:

- Summer peaks (weeks 20–35) in most species' curves reflect peak
  iNaturalist user activity, not necessarily peak pollinator activity.
- SDM-derived curves (Dan Cher) replace these for the SDM comparison
  experiments, providing model-predicted activity independent of
  observation effort.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

BASE        = Path("/scratch/ariana.l")
NEW_S4      = BASE / "New Stage 4 Link Prediction Model"
A_CURVES_IN = NEW_S4 / "a_curves_corrected.csv"

print("Paths OK")

In [ ]:
# Load a_curves
print("Loading a_curves...")
a_curves = pd.read_csv(A_CURVES_IN, index_col=0)
a_curves.columns = list(range(52))

print(f"  Shape: {a_curves.shape}")
# Expected: (25466, 52)

print(f"  Sample species: {list(a_curves.index[:5])}")

# Verify normalization: each row should sum to ~1
row_sums = a_curves.sum(axis=1)
print(f"  Row sum (mean ± std): {row_sums.mean():.4f} ± {row_sums.std():.4f}")
print(f"  All rows sum to ~1: {(row_sums.between(0.999, 1.001)).all()}")

In [ ]:
# Distribution of peak activity weeks
peak_weeks = a_curves.values.argmax(axis=1)
print("Peak activity week distribution:")
print(f"  Mean peak week: {peak_weeks.mean():.1f}")
print(f"  Median peak week: {np.median(peak_weeks):.1f}")
print()
print("Note: most species peak between weeks 20–35 (May–August).")
print("This matches peak iNaturalist user activity, not necessarily")
print("peak biological activity — observation-density bias.")

In [ ]:
# Plot mean activity curve across all species
weeks = np.arange(52)
mean_curve = a_curves.values.mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(weeks, mean_curve, color='steelblue', linewidth=2)
ax.fill_between(weeks, mean_curve, alpha=0.3, color='steelblue')
ax.set_xlabel('Week of year')
ax.set_ylabel('Mean normalized activity')
ax.set_title(
    'Mean GBIF activity curve across all 25,466 pollinator species\n'
    '(reflects observer effort — summer peak is iNaturalist recording pattern)'
)
ax.axvspan(20, 35, alpha=0.1, color='orange', label='Peak observer activity (weeks 20–35)')
ax.legend()
plt.tight_layout()
plt.savefig(NEW_S4 / 'a_curves_mean.png', dpi=150)
print("Saved a_curves_mean.png")

In [ ]:
# Coverage check: how many species have a_curves vs Vp
Vp_df = pd.read_csv(NEW_S4 / "stage4_Vp_corrected.csv", index_col=0)

a_species = set(a_curves.index)
vp_species = set(Vp_df.index)

print(f"Species in a_curves: {len(a_species):,}")
print(f"Species in Vp:       {len(vp_species):,}")
print(f"In both:             {len(a_species & vp_species):,}")
print(f"a_curves only:       {len(a_species - vp_species):,}")
print(f"Vp only:             {len(vp_species - a_species):,}")
print()
print("Species in both a_curves and Vp are the eligible pollinators")
print("for the model's shared pair universe.")